In [ ]:
# import all required libraries
import sys, os
import numpy as np
import pandas as pd
import random
from random import shuffle, choice
import time
import os
import glob
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.models import load_model
from tensorflow.keras import regularizers
from random import shuffle, choice
from sklearn.preprocessing import MinMaxScaler
import sklearn.metrics as metrics
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Model
from sklearn.preprocessing import MinMaxScaler,StandardScaler

# define a function to build a CNN for the SNP data.
def create_cnn(xtest, regularizer=None):
  # obtain the input dimensions.
  inputShape = (xtest.shape[1], xtest.shape[2])
  inputs = Input(shape=inputShape)
  x = inputs
  # first convolutional layer, remember to remove bias if you are intercalating with batch normalization.
  x = Conv1D(256, kernel_size=3, activation='relu', use_bias=False)(x)
  # batch normalization.
  x = BatchNormalization()(x)
  # second layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # third layer.
  x = Conv1D(256, kernel_size=3, use_bias=False, activation='relu')(x)
  x = BatchNormalization()(x)
  # pool the CNN outputs.
  x = GlobalMaxPooling1D()(x)
  # this part is similar to the MLP, a fully connected neural network. We intercalated with dropout to reduce overfitting.
  x = Dense(128, activation='relu')(x)
  # dropout.
  x = Dropout(0.5)(x)
  # second layer of the fully connected neural network.
  x = Dense(128, activation='relu')(x)
  x = Dropout(0.5)(x)
  # third layer of the fully connected neural network. This one matches the number of nodes coming out of the MLP.
  x = Dense(64, activation='relu')(x)
  # Construct the CNN
  #x = BatchNormalization()(x)#Not working very well
  #x = LayerNormalization()(x)#Better?
  model = Model(inputs, x)
  # Return the CNN
  return model

class GatedConcatenate(Layer):
    """
    Applies a trainable or fixed gate (weight) to each input branch
    before concatenating them.
    
    Args:
        initial_traits_weight (float): The starting weight for the first input (traits).
                                     Must be between 0 and 1. The weight for the
                                     second input (SNPs) will be (1 - this value).
        trainable_gates (bool): If True, the model can learn to adjust these
                                weights. If False, the weights are fixed.
    """
    def __init__(self, initial_traits_weight, trainable_gates=True, **kwargs):
        super(GatedConcatenate, self).__init__(**kwargs)
        if not (0 <= initial_traits_weight <= 1):
            raise ValueError("initial_traits_weight must be between 0 and 1.")
            
        self.initial_weights = [initial_traits_weight, 1.0 - initial_traits_weight]
        self.trainable_gates = trainable_gates

    def build(self, input_shape):
        # Create the gate variables. They are shaped for broadcasting across the features.
        self.gates = self.add_weight(
            name='gates',
            shape=(1, len(input_shape)), # Shape will be (1, 2)
            initializer=tf.constant_initializer(self.initial_weights),
            trainable=self.trainable_gates,
            dtype=tf.float32
        )
        super(GatedConcatenate, self).build(input_shape)

    def call(self, inputs):
        if not isinstance(inputs, list) or len(inputs) != 2:
            raise ValueError("GatedConcatenate expects a list of exactly two input tensors.")
        
        # Apply the gates (weights) to each branch using element-wise multiplication
        gated_traits = inputs[0] * self.gates[0, 0]
        gated_snps = inputs[1] * self.gates[0, 1]
        
        # Concatenate the scaled branches
        return Concatenate()([gated_traits, gated_snps])

    def get_config(self):
        # Needed for saving/loading the model
        config = super().get_config()
        config.update({
            'initial_traits_weight': self.initial_weights[0],
            'trainable_gates': self.trainable_gates,
        })
        return config
    
def gated_contributions(model, layer_name=None, labels=("traits", "SNPs")):
    # 1) find the layer
    gated_layer = model.get_layer('gated_concatenate')
    weights = gated_layer.get_weights()[0][0]
    rel_weight = np.sum(np.abs(weights))
    print(f"Final learned weights: Traits={weights[0]/rel_weight:.4f}, SNPs={weights[1]/rel_weight:.4f}")

In [3]:
## define variables that will be used to train all networks.
# size of the minibatches containing simulations are passed through the network in each epoch.
batch_size = 256
# number of training iterations (epochs) for the SNP only and the combined networks.
epochs = 100
# number of training iterations (epochs) for the traits only networks.
num_classes = 5

Load data
###############################################################################################################################################

In [ ]:
traits_BM = []
#c_traits = np.stack([np.loadtxt(f) for f in all_files])
traits_BM = np.loadtxt("./traits/traits_BM.txt").reshape(50000,-1,4)

traits_BM = np.array(traits_BM)

#Use standard scaling for the continuous traits.
scalers_BM = {}
for i in range(traits_BM.shape[2]):
    scalers_BM[i] = StandardScaler(copy=False)
    traits_BM[:, :, i] = scalers_BM[i].fit_transform(traits_BM[:, :, i]) 

traits_BM = np.array(traits_BM)


traits_OU = []
traits_OU = np.loadtxt("./traits/traits_OU.txt").reshape(50000,-1,4)

traits_OU = np.array(traits_OU)

#Use standard scaling for the continuous traits
scalers_OU = {}
for i in range(traits_OU.shape[2]):
    scalers_OU[i] = StandardScaler(copy=False)
    traits_OU[:, :, i] = scalers_OU[i].fit_transform(traits_OU[:, :, i]) 

traits_OU = np.array(traits_OU)

#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_BM.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_BM[i,j,:] = 0
    
#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_OU.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_OU[i,j,:] = 0

u1 = np.load("./trainingSims/Model_1sp.npz")
u2 = np.load("./trainingSims/Model_2spMorph.npz")
u3 = np.load("./trainingSims/Model_2spPhylo.npz")
u4 = np.load("./trainingSims/Model_3sp.npz")
u5 = np.load("./trainingSims/Model_3sp_M.npz")

#change to Model_3sp_M
x=np.concatenate((u1['Model_1sp'],u2['Model_2spMorph'],u3['Model_2spPhylo'],u4['Model_3sp'],u5['Model_3sp_M']),axis=0)

#transform major alleles in -1 and minor 1
for arr,array in enumerate(x):
  for idx,row in enumerate(array):
    if np.count_nonzero(row) > len(row)/2:
      x[arr][idx][x[arr][idx] == 1] = -1
      x[arr][idx][x[arr][idx] == 0] = 1
    else:
      x[arr][idx][x[arr][idx] == 0] = -1

y=[0 for i in range(len(u1['Model_1sp']))]
y.extend([1 for i in range(len(u2['Model_2spMorph']))])
y.extend([2 for i in range(len(u3['Model_2spPhylo']))])
y.extend([3 for i in range(len(u4['Model_3sp']))])
y.extend([4 for i in range(len(u5['Model_3sp_M']))])
y = np.array(y)

#Add missing data as 0s, according to a specifies missing data percentage
missD_perc = 17.7
missD = int(x.shape[1]*x.shape[2]*(missD_perc/100))
for i in range(x.shape[0]):
  for m in range(missD):
    j = random.randint(0, x.shape[1] - 1)
    k = random.randint(0, x.shape[2] - 1)
    x[i][j][k] = 0

del(missD)


Train the combined SNPs + traits (BM) network
###############################################################################################################################################

We will start with traits simulated under the BM model.
###############################################################################################################################################

In [ ]:
################################################################################################################################################
# We will start with traits simulated under the BM model.
################################################################################################################################################

# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and BM traits only).

# function to train on the combined datasets
def combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_BM_train)
    snps = create_cnn(xtrain)

    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_BM_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_BM_test, xtest], ytest),callbacks=[earlyStopping])

    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')

    return model

# function to train on the SNP only datasets
def SNP_subset(ytrain, ytest, xtrain, xtest):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]

    # Create the the CNN 
    snps = create_cnn(xtest)
    
    #Create the last layer for the SNP network
    xSNP = Dense(num_classes, activation="softmax")(snps.output)
    model = Model(inputs=snps.input, outputs=xSNP)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit(xtrain, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(xtest, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the BM trait only datasets
def BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_BM_train=np.swapaxes(traits_BM_train, 1, 2)
    traits_BM_test=np.swapaxes(traits_BM_test, 1, 2)
    trait = create_cnn(traits_BM_train)
    
    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_BM_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_BM_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')

    return model

4 BM, 1,000 SNPs
###############################################################################################################################################
separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.

In [10]:
ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test  = train_test_split(y,x,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model1 = combined_BM_subset(ytrain, ytest, xtrain, xtest, traits_BM_train, traits_BM_test)

model1.save(filepath='./TrainedModels/Trained_Comb_Model_1KSNPs_BM.acc.mod')

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 4, 109)]     0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 428, 109)]   0                                            
__________________________________________________________________________________________________
conv1d (Conv1D)                 (None, 4, 256)       83712       input_1[0][0]                    
__________________________________________________________________________________________________
conv1d_3 (Conv1D)               (None, 428, 256)     83712       input_2[0][0]                    
____________________________________________________________________________________________

Epoch 14/100
147/147 [==============================] - 15s 104ms/step - loss: 0.2856 - accuracy: 0.8286 - val_loss: 0.2725 - val_accuracy: 0.8349
Epoch 15/100
147/147 [==============================] - 15s 102ms/step - loss: 0.2837 - accuracy: 0.8314 - val_loss: 0.2710 - val_accuracy: 0.8365
Epoch 16/100
147/147 [==============================] - 15s 103ms/step - loss: 0.2826 - accuracy: 0.8316 - val_loss: 0.2712 - val_accuracy: 0.8372
Epoch 17/100
147/147 [==============================] - 18s 122ms/step - loss: 0.2773 - accuracy: 0.8359 - val_loss: 0.2707 - val_accuracy: 0.8381
Epoch 18/100
147/147 [==============================] - 16s 110ms/step - loss: 0.2754 - accuracy: 0.8371 - val_loss: 0.2705 - val_accuracy: 0.8348
Epoch 19/100
147/147 [==============================] - 18s 122ms/step - loss: 0.2734 - accuracy: 0.8395 - val_loss: 0.2695 - val_accuracy: 0.8376
Epoch 20/100
147/147 [==============================] - 18s 126ms/step - loss: 0.2703 - accuracy: 0.8438 - val_loss: 0

Epoch 70/100
147/147 [==============================] - 18s 123ms/step - loss: 0.0583 - accuracy: 0.9793 - val_loss: 0.5314 - val_accuracy: 0.8323
Epoch 71/100
147/147 [==============================] - 17s 113ms/step - loss: 0.0614 - accuracy: 0.9789 - val_loss: 0.6996 - val_accuracy: 0.8285
Epoch 72/100
147/147 [==============================] - 18s 121ms/step - loss: 0.0572 - accuracy: 0.9806 - val_loss: 0.5590 - val_accuracy: 0.8325
Final learned weights: Traits=0.2697, SNPs=0.7303
INFO:tensorflow:Assets written to: ./TrainedModels/Trained_Comb_Model_1KSNPs_BM.acc.mod/assets


Train the SNPs only network
###############################################################################################################################################

1,000 SNPs
###############################################################################################################################################

In [11]:
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, xtrain, xtest = train_test_split(y,x,test_size=0.25, shuffle=True,stratify=y)

# train the network
model2 = SNP_subset(ytrain, ytest, xtrain, xtest)

model2.save(filepath='./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod')

Model: "model_4"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         [(None, 428, 109)]        0         
_________________________________________________________________
conv1d_6 (Conv1D)            (None, 428, 256)          83712     
_________________________________________________________________
batch_normalization_6 (Batch (None, 428, 256)          1024      
_________________________________________________________________
conv1d_7 (Conv1D)            (None, 428, 256)          196608    
_________________________________________________________________
batch_normalization_7 (Batch (None, 428, 256)          1024      
_________________________________________________________________
conv1d_8 (Conv1D)            (None, 428, 256)          196608    
_________________________________________________________________
batch_normalization_8 (Batch (None, 428, 256)          1024

Epoch 42/100
147/147 [==============================] - 18s 120ms/step - loss: 0.2831 - accuracy: 0.8112 - val_loss: 0.2748 - val_accuracy: 0.8117
Epoch 43/100
147/147 [==============================] - 18s 119ms/step - loss: 0.2815 - accuracy: 0.8098 - val_loss: 0.2744 - val_accuracy: 0.8277
Epoch 44/100
147/147 [==============================] - 18s 121ms/step - loss: 0.2813 - accuracy: 0.8150 - val_loss: 0.2744 - val_accuracy: 0.8321
Epoch 45/100
147/147 [==============================] - 16s 107ms/step - loss: 0.2813 - accuracy: 0.8131 - val_loss: 0.2751 - val_accuracy: 0.8198
Epoch 46/100
147/147 [==============================] - 18s 120ms/step - loss: 0.2806 - accuracy: 0.8158 - val_loss: 0.2746 - val_accuracy: 0.8271
Epoch 47/100
147/147 [==============================] - 17s 118ms/step - loss: 0.2803 - accuracy: 0.8149 - val_loss: 0.2736 - val_accuracy: 0.8106
Epoch 48/100
147/147 [==============================] - 18s 121ms/step - loss: 0.2798 - accuracy: 0.8155 - val_loss: 0

Epoch 98/100
147/147 [==============================] - 19s 127ms/step - loss: 0.0605 - accuracy: 0.9742 - val_loss: 0.6674 - val_accuracy: 0.8303
Epoch 99/100
147/147 [==============================] - 19s 127ms/step - loss: 0.0634 - accuracy: 0.9725 - val_loss: 0.6694 - val_accuracy: 0.8274
Epoch 100/100
147/147 [==============================] - 19s 127ms/step - loss: 0.0537 - accuracy: 0.9780 - val_loss: 0.6659 - val_accuracy: 0.8325
Time: 1836.1398162841797
INFO:tensorflow:Assets written to: ./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod/assets


Train the traits only (BM) network
###############################################################################################################################################

4 BM
###############################################################################################################################################

In [12]:
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_BM_train, traits_BM_test  = train_test_split(y,traits_BM,test_size=0.25, shuffle=True,stratify=y)

# train the network
model3 = BM_subset(ytrain, ytest, traits_BM_train, traits_BM_test)
          
model3.save(filepath='./TrainedModels/Trained_Traits_Model_BM.acc.mod')


Model: "model_6"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_4 (InputLayer)         [(None, 4, 109)]          0         
_________________________________________________________________
conv1d_9 (Conv1D)            (None, 4, 256)            83712     
_________________________________________________________________
batch_normalization_9 (Batch (None, 4, 256)            1024      
_________________________________________________________________
conv1d_10 (Conv1D)           (None, 4, 256)            196608    
_________________________________________________________________
batch_normalization_10 (Batc (None, 4, 256)            1024      
_________________________________________________________________
conv1d_11 (Conv1D)           (None, 4, 256)            196608    
_________________________________________________________________
batch_normalization_11 (Batc (None, 4, 256)            1024

Train the combined SNPs + traits (OU) network
###############################################################################################################################################

Now repeat with traits simulated under the OU model.
###############################################################################################################################################

In [ ]:
# Since we will run the analysis on several subsets, define a function for training on each data subsets (Combined datasets, SNP only and OU traits only).

# function to train on the combined datasets
def combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    # Create the two CNNs and the combined models
    traits = create_cnn(traits_OU_train)
    snps = create_cnn(xtrain)

    # Use the gated concatenation layer. Start with an 50/50 contribution for each branch, but let the model learn.
    # To set pre-defined weights for each branch, change here  the "initial_traits_weight" to define the traits relative contribution (from 0 to 1).
    # To keep the weight of each branch fixed, change "trainable_gates" to false.
    combinedInput = GatedConcatenate(
        initial_traits_weight=0.5, 
        trainable_gates=True,
        name="gated_concatenate"
    )([traits.output, snps.output])

    # The final fully-connected layer head will have two dense layers (one relu and one softmax)
    x = Dense(64, activation="relu")(combinedInput)

    x = Dense(num_classes, activation="softmax")(x)

    # The final model accepts numerical data on the MLP input and images on the CNN input, outputting a single value
    model = Model(inputs=[traits.input, snps.input], outputs=x)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())
    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)

    # fit the model and record running times
    start = time.time()
    model.fit([traits_OU_train, xtrain], ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=([traits_OU_test, xtest], ytest),callbacks=[earlyStopping])
    
    # Get contributions from each branch.
    gated_contributions(model)
    print (f'Time: {time.time() - start}')
    
    return model

# function to train on the OU trait only datasets
def OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test):
    # convert labels to a categorical matrix of binary values (0 or 1). The number of rows is the length of the input vector (number of simulations) and the number of columns is the number of classes (3 scenarios).
    ytest = np.eye(num_classes)[ytest]
    ytrain = np.eye(num_classes)[ytrain]
    # swap traits axes to match SNPs (individuals as columns and each trait in a line)
    traits_OU_train=np.swapaxes(traits_OU_train, 1, 2)
    traits_OU_test=np.swapaxes(traits_OU_test, 1, 2)
    trait = create_cnn(traits_OU_train)

    #Create the last layer for the traits network
    xTRAIT = Dense(num_classes, activation="softmax")(trait.output)
    model = Model(inputs=trait.input, outputs=xTRAIT)

    # using Stochastic Gradient Descent as optimizer and a categorical cross-entropy loss function
    opt = SGD(learning_rate=0.001,momentum=0.9,nesterov=True)
    model.compile(loss=tf.keras.losses.categorical_crossentropy,
              optimizer=opt,
              metrics=['accuracy'])

    print(model.summary())

    # save only the epoch with the highest accuracy in the validation set, by using the model checkpoint
    earlyStopping = EarlyStopping(monitor='val_accuracy', patience=50, verbose=0, mode='max', restore_best_weights=True)
    # fit the model and record running times
    start = time.time()
    model.fit(traits_OU_train, ytrain, batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(traits_OU_test, ytest),callbacks=[earlyStopping])
    print (f'Time: {time.time() - start}')
    
    return model

4 OU, 1,000 SNPs
###############################################################################################################################################
separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.

In [15]:
ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test  = train_test_split(y,x,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model4 = combined_OU_subset(ytrain, ytest, xtrain, xtest, traits_OU_train, traits_OU_test)

model4.save(filepath='./TrainedModels/Trained_Comb_Model_1KSNPs_OU.acc.mod')

Model: "model_9"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_5 (InputLayer)            [(None, 4, 109)]     0                                            
__________________________________________________________________________________________________
input_6 (InputLayer)            [(None, 428, 109)]   0                                            
__________________________________________________________________________________________________
conv1d_12 (Conv1D)              (None, 4, 256)       83712       input_5[0][0]                    
__________________________________________________________________________________________________
conv1d_15 (Conv1D)              (None, 428, 256)     83712       input_6[0][0]                    
____________________________________________________________________________________________

Epoch 14/100
147/147 [==============================] - 19s 130ms/step - loss: 0.2926 - accuracy: 0.8069 - val_loss: 0.2774 - val_accuracy: 0.8051
Epoch 15/100
147/147 [==============================] - 19s 133ms/step - loss: 0.2913 - accuracy: 0.8077 - val_loss: 0.2772 - val_accuracy: 0.8056
Epoch 16/100
147/147 [==============================] - 19s 132ms/step - loss: 0.2892 - accuracy: 0.8059 - val_loss: 0.2800 - val_accuracy: 0.8005
Epoch 17/100
147/147 [==============================] - 20s 136ms/step - loss: 0.2882 - accuracy: 0.8051 - val_loss: 0.2766 - val_accuracy: 0.8114
Epoch 18/100
147/147 [==============================] - 19s 132ms/step - loss: 0.2872 - accuracy: 0.8083 - val_loss: 0.2779 - val_accuracy: 0.8029
Epoch 19/100
147/147 [==============================] - 19s 130ms/step - loss: 0.2852 - accuracy: 0.8127 - val_loss: 0.2770 - val_accuracy: 0.8082
Epoch 20/100
147/147 [==============================] - 19s 127ms/step - loss: 0.2859 - accuracy: 0.8131 - val_loss: 0

Epoch 70/100
147/147 [==============================] - 28s 188ms/step - loss: 0.0835 - accuracy: 0.9684 - val_loss: 0.4897 - val_accuracy: 0.8335
Epoch 71/100
147/147 [==============================] - 27s 182ms/step - loss: 0.0758 - accuracy: 0.9726 - val_loss: 0.5096 - val_accuracy: 0.8310
Epoch 72/100
147/147 [==============================] - 27s 181ms/step - loss: 0.0718 - accuracy: 0.9751 - val_loss: 0.5274 - val_accuracy: 0.8304
Epoch 73/100
147/147 [==============================] - 27s 181ms/step - loss: 0.0689 - accuracy: 0.9751 - val_loss: 0.7291 - val_accuracy: 0.8246
Epoch 74/100
147/147 [==============================] - 27s 186ms/step - loss: 0.0686 - accuracy: 0.9749 - val_loss: 0.5831 - val_accuracy: 0.8310
Epoch 75/100
147/147 [==============================] - 27s 183ms/step - loss: 0.0623 - accuracy: 0.9789 - val_loss: 0.5099 - val_accuracy: 0.8354
Epoch 76/100
147/147 [==============================] - 27s 182ms/step - loss: 0.0598 - accuracy: 0.9790 - val_loss: 0

Train the traits only (OU) network
###############################################################################################################################################
###############################################################################################################################################
4 OU
###############################################################################################################################################

In [7]:
# separate 75% of labels, SNP and traits matrices as training set. The other 25% are assigned to the test set. The two sets are shuffled.
ytrain, ytest, traits_OU_train, traits_OU_test  = train_test_split(y,traits_OU,test_size=0.25, shuffle=True,stratify=y)

# train the network
model5 = OU_subset(ytrain, ytest, traits_OU_train, traits_OU_test)
          
model5.save(filepath='./TrainedModels/Trained_Traits_Model_OU.acc.mod')

Model: "model_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 4, 109)]          0         
_________________________________________________________________
conv1d (Conv1D)              (None, 4, 256)            83712     
_________________________________________________________________
batch_normalization (BatchNo (None, 4, 256)            1024      
_________________________________________________________________
conv1d_1 (Conv1D)            (None, 4, 256)            196608    
_________________________________________________________________
batch_normalization_1 (Batch (None, 4, 256)            1024      
_________________________________________________________________
conv1d_2 (Conv1D)            (None, 4, 256)            196608    
_________________________________________________________________
batch_normalization_2 (Batch (None, 4, 256)            1024

147/147 [==============================] - 2s 11ms/step - loss: 0.3681 - accuracy: 0.7781 - val_loss: 1.0477 - val_accuracy: 0.6722
Epoch 43/100
147/147 [==============================] - 2s 11ms/step - loss: 0.3611 - accuracy: 0.7813 - val_loss: 1.0856 - val_accuracy: 0.6658
Epoch 44/100
147/147 [==============================] - 2s 11ms/step - loss: 0.3503 - accuracy: 0.7818 - val_loss: 1.1074 - val_accuracy: 0.6679
Epoch 45/100
147/147 [==============================] - 2s 11ms/step - loss: 0.3464 - accuracy: 0.7854 - val_loss: 1.1303 - val_accuracy: 0.6680
Epoch 46/100
147/147 [==============================] - 2s 11ms/step - loss: 0.3490 - accuracy: 0.7879 - val_loss: 1.1632 - val_accuracy: 0.6663
Epoch 47/100
147/147 [==============================] - 2s 11ms/step - loss: 0.3392 - accuracy: 0.7845 - val_loss: 1.1883 - val_accuracy: 0.6651
Epoch 48/100
147/147 [==============================] - 2s 11ms/step - loss: 0.3386 - accuracy: 0.7891 - val_loss: 1.1814 - val_accuracy: 0.662

#Perform cross-validation
################################################################################################################################################

In [ ]:

model1 = load_model('./TrainedModels/Trained_Comb_Model_1KSNPs_BM.acc.mod')
model2 = load_model('./TrainedModels/Trained_CNN_Model_1KSNPs.acc.mod')
model3 = load_model('./TrainedModels/Trained_Traits_Model_BM.acc.mod')
model4 = load_model('./TrainedModels/Trained_Comb_Model_1KSNPs_OU.acc.mod')
model5 = load_model('./TrainedModels/Trained_Traits_Model_OU.acc.mod')

traits_BM = []
traits_BM = np.loadtxt("./testSims/traits/traits_BM.txt").reshape(5000,-1,4)

traits_BM = np.array(traits_BM)
Traits_BM = np.array(traits_BM)

#Use standard scaling for the continuous traits.
for i in range(traits_BM.shape[2]):
    traits_BM[:, :, i] = scalers_BM[i].transform(traits_BM[:, :, i]) 

traits_BM = np.array(traits_BM)


traits_OU = []
traits_OU = np.loadtxt("./testSims/traits/traits_OU.txt").reshape(5000,-1,4)

traits_OU = np.array(traits_OU)

#Use standard scaling for the continuous traits.
for i in range(traits_OU.shape[2]):
    traits_OU[:, :, i] = scalers_OU[i].transform(traits_OU[:, :, i]) 

traits_OU = np.array(traits_OU)

#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_BM.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_BM[i,j,:] = 0
    
#Add missing individuals in E. balsamifera subsp. balsamifera as 0s
for i in range(traits_OU.shape[0]):
  for m in range(51):
    j = random.randint(19, 98)
    traits_OU[i,j,:] = 0

u1 = np.load("./testSims/Model_1sp.npz")
u2 = np.load("./testSims/Model_2spMorph.npz")
u3 = np.load("./testSims/Model_2spPhylo.npz")
u4 = np.load("./testSims/Model_3sp.npz")
u5 = np.load("./testSims/Model_3sp_M.npz")

xtest=np.concatenate((u1['Model_1sp'],u2['Model_2spMorph'],u3['Model_2spPhylo'],u4['Model_3sp'],u5['Model_3sp_M']),axis=0)

#transform major alleles in -1 and minor 1
for arr,array in enumerate(xtest):
  for idx,row in enumerate(array):
    if np.count_nonzero(row) > len(row)/2:
      xtest[arr][idx][xtest[arr][idx] == 1] = -1
      xtest[arr][idx][xtest[arr][idx] == 0] = 1
    else:
      xtest[arr][idx][xtest[arr][idx] == 0] = -1

#Add missing data as 0s, according to a specifies missing data percentage
#46,761 SNP datapoints and 8,287 missing genotypes = 17.7%
missD_perc = 17.7
missD = int(xtest.shape[1]*xtest.shape[2]*(missD_perc/100))
for i in range(xtest.shape[0]):
  for m in range(missD):
    j = random.randint(0, xtest.shape[1] - 1)
    k = random.randint(0, xtest.shape[2] - 1)
    xtest[i][j][k] = 0


ytest=[0 for i in range(len(u1['Model_1sp']))]
ytest.extend([1 for i in range(len(u2['Model_2spMorph']))])
ytest.extend([2 for i in range(len(u3['Model_2spPhylo']))])
ytest.extend([3 for i in range(len(u4['Model_3sp']))])
ytest.extend([4 for i in range(len(u5['Model_3sp_M']))])
ytest = np.array(ytest)

#first get the predictions
pred = model1.predict([np.swapaxes(traits_BM,1,2), xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model2.predict(xtest)
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model3.predict(np.swapaxes(traits_BM,1,2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model4.predict([np.swapaxes(traits_OU,1,2), xtest])
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))

#first get the predictions
pred = model5.predict(np.swapaxes(traits_OU,1,2))
pred_cat = [i.argmax() for i in pred]
ytest_cat = [i.argmax() for i in ytest]
print (confusion_matrix(ytest, pred_cat))


[[1000    0    0    0    0]
 [   0 1000    0    0    0]
 [   0    0 1000    0    0]
 [   0    2    0  771  227]
 [   0    0    1  538  461]]
[[1000    0    0    0    0]
 [   0 1000    0    0    0]
 [   0    0 1000    0    0]
 [   0    1    0  648  351]
 [   0    0    1  457  542]]
[[896  39  57   7   1]
 [ 54 858   5  67  16]
 [ 69   6 846  61  18]
 [ 16  85  94 703 102]
 [ 15  78  89 581 237]]
[[1000    0    0    0    0]
 [   0 1000    0    0    0]
 [   0    0 1000    0    0]
 [   0    1    1  671  327]
 [   0    0    0  447  553]]
[[902  40  46  11   1]
 [ 68 846   4  65  17]
 [100  11 798  68  23]
 [ 17 113 104 417 349]
 [ 12 117  83 405 383]]


Predict empirical data
###############################################################################################################################################

In [ ]:
# Load the SNP data set
inSNPs=np.loadtxt("./input_SNPs.txt")
EmpSNPs = np.array(inSNPs)

#transform major alleles in -1 and minor 1
for idx,row in enumerate(EmpSNPs):
  if np.count_nonzero(row==1) > np.count_nonzero(row==-1):
    EmpSNPs[idx][EmpSNPs[idx] == 1] = 9
    EmpSNPs[idx][EmpSNPs[idx] == -1] = 1
    EmpSNPs[idx][EmpSNPs[idx] == 9] = -1

EmpSNPs = np.repeat(EmpSNPs[np.newaxis, :, :], 100, axis=0)

#load the trait data
ade=np.genfromtxt("./input_traits_ade.txt", delimiter="\t", filling_values=0)
bal=np.genfromtxt("./input_traits_bal.txt", delimiter="\t", filling_values=0)
sep=np.genfromtxt("./input_traits_sep.txt", delimiter="\t", filling_values=0)

ade=np.array(ade)
bal=np.array(bal)
sep=np.array(sep)

#resample the trait data to build 100 bootstrap replicates
res = []
for i in range(0,100):
  idx_ade = np.random.choice(ade.shape[0], 19, replace=False)
  n = ade[idx_ade,:]
  idx_bal = np.random.choice(bal.shape[0], 29, replace=False)
  n_z = np.zeros((80,4))
  for i in range(len(idx_bal)):
    idx_nz=np.random.choice(80, len(idx_bal), replace=False)
    n_z[idx_nz[i]] = bal[idx_bal[i],:]
  n = np.concatenate((n,n_z), axis=0)
  idx_sep = np.random.choice(sep.shape[0], 10, replace=False)
  n = np.concatenate((n,sep[idx_sep,:]), axis=0)
  res.append(np.array(n))

traits = np.array(res)

emp_traits = np.array(traits)

#Use standard scaling for the continuous traits.
for i in range(emp_traits.shape[2]):
    emp_traits[:, :, i] = scalers_BM[i].transform(emp_traits[:, :, i]) 


Emp_comb_BM_pred = model1.predict([np.swapaxes(emp_traits,1,2),EmpSNPs])

np.savetxt("Pred_Emp_Comb_BM_Predictions.txt", Emp_comb_BM_pred)

Emp_SNP_pred = model2.predict(EmpSNPs)

np.savetxt("Pred_Emp_SNP_Predictions.txt", Emp_SNP_pred)

Emp_BM_traits_pred = model3.predict(np.swapaxes(emp_traits,1,2))

np.savetxt("Pred_Emp_traits_BM_Predictions.txt", Emp_BM_traits_pred)

emp_traits = np.array(traits)

#Use standard scaling for the continuous traits.
for i in range(emp_traits.shape[2]):
    emp_traits[:, :, i] = scalers_OU[i].transform(emp_traits[:, :, i]) 
    
Emp_comb_OU_pred = model4.predict([np.swapaxes(emp_traits,1,2),EmpSNPs])

np.savetxt("Pred_Emp_Comb_OU_Predictions.txt", Emp_comb_OU_pred)

Emp_OU_traits_pred = model5.predict(np.swapaxes(emp_traits,1,2))

np.savetxt("Pred_Emp_traits_OU_Predictions.txt", Emp_OU_traits_pred)